Purpose

Build a structured, validated evidence base across all available datasets, ensuring they are consistent, reliable, and ready for modelling. This notebook focuses on auditing data quality, structure, and compatibility before any transformation or estimation.

In [1]:
# Set-up and data loading 
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [2]:
# Project paths
PROJECT_DIR = Path.cwd()
print(f"Current working directory: {PROJECT_DIR}")

Current working directory: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities


In [3]:
# File names from project folder 
DATASETS = {
    "fingertips_nwl": PROJECT_DIR / "_processed_fingertips_nwl.parquet",
    "fingertips_selected_indicators": PROJECT_DIR / "_processed_fingertips_nwl_selected_indicators.parquet",
    "fingertips_indicator_metadata": PROJECT_DIR / "_processed_fingertips_indicator_metadata.parquet",
    "gla_his_indicators": PROJECT_DIR / "_processed_gla_his_indicators.parquet",
    "gla_his_borough_latest_and_trend": PROJECT_DIR / "_processed_gla_his_borough_latest_and_trend.parquet",
    "census2021_ethnicity_nwl_analytical": PROJECT_DIR / "_processed_census2021_ethnicity_nwl_analytical.parquet",
    "ethnicity_facts_gp_satisfaction": PROJECT_DIR / "_processed_ethnicity_facts_gp_satisfaction.parquet",
    "gpps_survey": PROJECT_DIR / "_processed_gpps_survey.parquet",
    "gpps_ethnicity_distribution": PROJECT_DIR / "_processed_gpps_ethnicity_distribution.parquet",
    "ons_birth_characteristics_national": PROJECT_DIR / "_processed_ons_birth_characteristics_national.parquet",
}

CONFIG_FILES = {
    "ethnicity_mapping": PROJECT_DIR / "_config_ethnicity_mapping.csv",
    "nwl_boroughs": PROJECT_DIR / "_config_nwl_boroughs.csv",
}


In [4]:
def check_file_existence(file_dict: dict) -> pd.DataFrame:
    rows = []

    for file_group, files in [("dataset", DATASETS), ("config", CONFIG_FILES)]:
        for name, path in files.items():
            rows.append(
                {
                    "file_group": file_group,
                    "name": name,
                    "path": str(path),
                    "exists": path.exists(),
                }
            )

    return pd.DataFrame(rows).sort_values(["file_group", "name"]).reset_index(drop=True)


file_check_df = check_file_existence(DATASETS)
file_check_df

,file_group,name,path,exists
0,config,ethnicity_mapping,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_config_e...,True
1,config,nwl_boroughs,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_config_n...,True
2,dataset,census2021_ethnicity_nwl_analytical,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
3,dataset,ethnicity_facts_gp_satisfaction,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
4,dataset,fingertips_indicator_metadata,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
5,dataset,fingertips_nwl,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
6,dataset,fingertips_selected_indicators,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
7,dataset,gla_his_borough_latest_and_trend,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
8,dataset,gla_his_indicators,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True
9,dataset,gpps_ethnicity_distribution,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True


In [5]:
# Robust parquet loader with audit logging
def load_parquet_datasets(dataset_paths: dict) -> tuple[dict, pd.DataFrame]:
    loaded_datasets = {}
    audit_rows = []

    for dataset_name, file_path in dataset_paths.items():
        audit_row = {
            "dataset_name": dataset_name,
            "file_name": file_path.name,
            "path": str(file_path),
            "exists": file_path.exists(),
            "loaded": False,
            "rows": None,
            "columns": None,
            "status": None,
            "error_type": None,
            "error_message": None,
        }

        if not file_path.exists():
            audit_row["status"] = "missing_file"
            audit_row["error_type"] = "FileNotFoundError"
            audit_row["error_message"] = "File does not exist."
            audit_rows.append(audit_row)
            print(f"[Missing] {dataset_name}: {file_path.name}")
            continue

        try:
            dataframe = pd.read_parquet(file_path)
            loaded_datasets[dataset_name] = dataframe

            audit_row["loaded"] = True
            audit_row["rows"] = int(dataframe.shape[0])
            audit_row["columns"] = int(dataframe.shape[1])
            audit_row["status"] = "loaded"

            print(
                f"[Loaded] {dataset_name}: "
                f"{dataframe.shape[0]:,} rows × {dataframe.shape[1]:,} columns"
            )

        except Exception as error:
            audit_row["status"] = "read_error"
            audit_row["error_type"] = type(error).__name__
            audit_row["error_message"] = str(error)

            print(
                f"[Failed] {dataset_name}: "
                f"{file_path.name} -> {type(error).__name__}: {error}"
            )

        audit_rows.append(audit_row)

    audit_df = pd.DataFrame(audit_rows).sort_values(
        ["loaded", "dataset_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return loaded_datasets, audit_df

Part 1 - Data loading & audit

Load all processed datasets and verify availability, integrity, and successful ingestion.

In [6]:
# Load all parquet datasets with audit
datasets, dataset_load_audit = load_parquet_datasets(DATASETS)

print(f"\nLoaded {len(datasets)} dataset(s) out of {len(DATASETS)} expected.")
print("Loaded dataset keys:")
print(list(datasets.keys()))

dataset_load_audit

[Failed] fingertips_nwl: _processed_fingertips_nwl.parquet -> ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes
[Loaded] fingertips_selected_indicators: 5,118 rows × 28 columns
[Loaded] fingertips_indicator_metadata: 1,008 rows × 32 columns
[Failed] gla_his_indicators: _processed_gla_his_indicators.parquet -> ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes
[Loaded] gla_his_borough_latest_and_trend: 3,611 rows × 14 columns
[Loaded] census2021_ethnicity_nwl_analytical: 64 rows × 6 columns
[Loaded] ethnicity_facts_gp_satisfaction: 21 rows × 11 columns
[Loaded] gpps_survey: 630 rows × 11 columns
[Loaded] gpps_ethnicity_distribution: 252 rows × 12 columns
[Loaded] ons_birth_characteristics_national: 995 rows × 13 columns

Loaded 8 dataset(s) out of 10 expected.
Loaded dataset keys:
['fingertips_selected_indicators', 'fingertips_indicator_metadata', 'gla_his_borough_latest_and_trend', 'census2021_ethnicity_nw

,dataset_name,file_name,path,exists,loaded,rows,columns,status,error_type,error_message
0,census2021_ethnicity_nwl_analytical,_processed_census2021_ethnicity_nwl_analytical.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,64.0,6.0,loaded,NaN,NaN
1,ethnicity_facts_gp_satisfaction,_processed_ethnicity_facts_gp_satisfaction.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,21.0,11.0,loaded,NaN,NaN
2,fingertips_indicator_metadata,_processed_fingertips_indicator_metadata.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,1008.0,32.0,loaded,NaN,NaN
3,fingertips_selected_indicators,_processed_fingertips_nwl_selected_indicators.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,5118.0,28.0,loaded,NaN,NaN
4,gla_his_borough_latest_and_trend,_processed_gla_his_borough_latest_and_trend.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,3611.0,14.0,loaded,NaN,NaN
5,gpps_ethnicity_distribution,_processed_gpps_ethnicity_distribution.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,252.0,12.0,loaded,NaN,NaN
6,gpps_survey,_processed_gpps_survey.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,630.0,11.0,loaded,NaN,NaN
7,ons_birth_characteristics_national,_processed_ons_birth_characteristics_national.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,True,995.0,13.0,loaded,NaN,NaN
8,fingertips_nwl,_processed_fingertips_nwl.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,False,NaN,NaN,read_error,ArrowInvalid,Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes
9,gla_his_indicators,_processed_gla_his_indicators.parquet,c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_processe...,True,False,NaN,NaN,read_error,ArrowInvalid,Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes


In [7]:
# Dataset registry
def build_dataset_registry(datasets_dict: dict) -> pd.DataFrame:
    registry_rows = []

    for dataset_name, dataframe in datasets_dict.items():
        registry_rows.append(
            {
                "dataset_name": dataset_name,
                "rows": int(dataframe.shape[0]),
                "columns": int(dataframe.shape[1]),
                "column_names": list(dataframe.columns),

                "has_table": "table" in dataframe.columns,
                "has_source": "source" in dataframe.columns,
                "has_year": "year" in dataframe.columns,
                "has_geography": "geography" in dataframe.columns,
                "has_area_code": "area_code" in dataframe.columns,
                "has_area_name": "area_name" in dataframe.columns,
                "has_category": "category" in dataframe.columns,
                "has_metric": "metric" in dataframe.columns,
                "has_ethnicity_standard": "ethnicity_standard" in dataframe.columns,
                "has_measure_raw": "measure_raw" in dataframe.columns,
                "has_value": "value" in dataframe.columns,
            }
        )

    registry_df = pd.DataFrame(registry_rows)

    if not registry_df.empty:
        registry_df = registry_df.sort_values("dataset_name").reset_index(drop=True)

    return registry_df

In [8]:
# Build dataset registry
dataset_registry = build_dataset_registry(datasets)

print(f"Registry created for {len(dataset_registry)} dataset(s)")
dataset_registry

Registry created for 8 dataset(s)


,dataset_name,rows,columns,column_names,has_table,has_source,has_year,has_geography,has_area_code,has_area_name,has_category,has_metric,has_ethnicity_standard,has_measure_raw,has_value
0,census2021_ethnicity_nwl_analytical,64,6,"[geography_code, geography_name, analytical_ethnicity, population, borough_total, proportion]",False,False,False,False,False,False,False,False,False,False,False
1,ethnicity_facts_gp_satisfaction,21,11,"[source, dataset, geography_level, geography_name, time_period, measure_name, value, value_unit, raw_ethnicity, stan...",False,True,False,False,False,False,False,False,False,False,True
2,fingertips_indicator_metadata,1008,32,"[Indicator ID, Indicator, Indicator number, Rationale, Specific rationale, Definition, Data source, Indicator source...",False,False,False,False,False,False,False,False,False,False,False
3,fingertips_selected_indicators,5118,28,"[Indicator ID, Indicator Name, Parent Code, Parent Name, Area Code, Area Name, Area Type, Sex, Age, Category Type, C...",False,False,False,False,False,False,False,False,False,False,False
4,gla_his_borough_latest_and_trend,3611,14,"[time_period, borough, value, lower_ci, upper_ci, neg_error_bar, pos_error_bar, indicator_group, data_version, , Neg...",False,False,False,False,False,False,False,False,False,False,True
5,gpps_ethnicity_distribution,252,12,"[table, source, year, geography, area_geography, area_code, area_name, metric, ethnicity_raw, ethnicity_standard, me...",True,True,True,True,True,True,False,True,True,True,True
6,gpps_survey,630,11,"[table, source, year, geography, area_geography, area_code, area_name, metric, category, measure_raw, value]",True,True,True,True,True,True,True,True,False,True,True
7,ons_birth_characteristics_national,995,13,"[table, source, year, geography, area_geography, area_code, area_name, category, metric, ethnicity_raw, ethnicity_st...",True,True,True,True,True,True,True,True,True,True,True


In [9]:
# Save dataset registry

dataset_registry.to_csv(PROJECT_DIR / "_notebook10a_dataset_registry.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_dataset_registry.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_dataset_registry.csv


In [10]:
# Preview first 3 rows of each dataset
for dataset_name, dataframe in datasets.items():
    print("=" * 100)
    print(dataset_name)
    print("=" * 100)
    display(dataframe.head(3))

fingertips_selected_indicators


,Indicator ID,Indicator Name,Parent Code,Parent Name,Area Code,Area Name,Area Type,Sex,Age,Category Type,Category,Time period,Value,Lower CI 95.0 limit,Upper CI 95.0 limit,Lower CI 99.8 limit,Upper CI 99.8 limit,Count,Denominator,Value note,Recent Trend,Compared to England value or percentiles,Column not used,Time period Sortable,New data,Compared to goal,Time period range,indicator_id
0,224,Epilepsy: QOF prevalence,E92000001,England,E09000005,Brent,UA,Persons,18+ yrs,NaN,<NA>,2012/13,0.535821,0.509782,0.562880,0.495463,0.579247,1554.0,290022.328867,NaN,<NA>,Lower 99.8,Not compared,20120000,NaN,NaN,1y,224
1,224,Epilepsy: QOF prevalence,E92000001,England,E09000009,Ealing,UA,Persons,18+ yrs,NaN,<NA>,2012/13,0.546864,0.521516,0.573168,0.507464,0.589136,1694.0,309765.976377,NaN,<NA>,Lower 99.8,Not compared,20120000,NaN,NaN,1y,224
2,224,Epilepsy: QOF prevalence,E92000001,England,E09000013,Hammersmith and Fulham,UA,Persons,18+ yrs,NaN,<NA>,2012/13,0.524154,0.490395,0.559713,0.471833,0.581887,862.0,164455.457809,There is a data quality issue with this value,<NA>,Lower 99.8,Not compared,20120000,NaN,NaN,1y,224


fingertips_indicator_metadata


,Indicator ID,Indicator,Indicator number,Rationale,Specific rationale,Definition,Data source,Indicator source,Definition of numerator,Source of numerator,Definition of denominator,Source of denominator,Methodology,Standard population/values,Frequency,Confidence interval details,Disclosure control,Rounding,Caveats,Notes,Impact of COVID-19,Copyright,Data re-use,Links,Indicator Content,Simple Name,Simple Definition,Unit,Value type,Year type,Polarity,Date updated
0,108,Under 75 mortality rate from all causes,NaN,"Premature mortality is a good high-level indicator of the overall health of a population, being correlated with many...",NaN,"Directly age-standardised mortality rate for all deaths, per 100,000 population, in those aged under 75 years","OHID, based on Office for National Statistics data",Office for Health Improvement and Disparities,"Number of deaths registered in the respective calendar years, in people aged under 75, aggregated into quinary age b...","Office for National Statistics (ONS), Annual mortality extract (produced for OHID)","For single years: Population for people under 75 years of age, aggregated into quinary age bands (0 to 4, 5 to 9, …,...","Office for National Statistics (ONS), Mid-year population estimates",Numerator data for each age band are divided by the denominator population data for each age band respectively to gi...,2013 European Standard Population,The data will be updated annually,NaN,"For ICBs that are almost, but not precisely, coterminous with local authorities, the local authority boundaries have...",NaN,Please note data may not match published Office for National Statistics (ONS) figures due to differences in the vers...,"Population estimates produced by the Office for National Statistics for mid-2022 and mid-2023 have been revised, to ...",This indicator covers the period since March 2020 and includes the impact of the COVID-19 pandemic,© Crown copyright,Please cite any use of this website as follows specifying the date of access: \r\nOffice for Health Improvement and ...,NaN,"Directly age-standardised rate of mortality from all causes, under 75s","Deaths from all causes, ages under 75",Rate of deaths from all causes for people aged under 75,"per 100,000",Directly standardised rate,Calendar,RAG - Low is good,13/11/2025
1,114,QOF Total List Size,NaN,NaN,NaN,Total number of patients registered with the practice,"NHS England, Quality and Outcomes Framework","For all QOF data for the current and for previous years, please see the https://digital.nhs.uk/data-and-information...",NaN,"NHS England (NHSE), Quality and Outcomes Framework (QOF)",NaN,Not applicable (N/A),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Count,Financial,Not applicable,10/12/2025
2,200,Learning disability: QOF prevalence,NaN,NaN,NaN,"The percentage of patients with learning disabilities, as recorded on practice disease registers",NHS England,"For all QOF data for the current and for previous years, please see the https://digital.nhs.uk/data-and-information...","Total number of patients with learning disabilities, as recorded on the practice disease register.","NHS England (NHSE), Quality and Outcomes Framework (QOF)",Total number of patients registered with the practice.,"NHS England (NHSE), Quality and Outcomes Framework (QOF)",numerator/denominator * 100,NaN,NaN,NaN,NaN,NaN,NaN,"Where local authority values are presented, these were calculated by assigning all patients of the GP to the local a...","For 2020/21 QOF data, https://digital.nhs.uk/data-and-information/publications/statistical/quality-and-outcomes-fra...",NaN,NaN,NaN,NaN,NaN,NaN,%,Proportion,Financial,BOB - Blue orange blue,10/12/2025


gla_his_borough_latest_and_trend


...

,table,source,year,geography,area_geography,area_code,area_name,metric,category,measure_raw,value
0,GPPS_ICS,GP Patient Survey,2025,England,ICS,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE SYSTEM,gpcontactoverall,1,gpcontactoverall_1.pct,0.425938
1,GPPS_ICS,GP Patient Survey,2025,England,ICS,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,gpcontactoverall,1,gpcontactoverall_1.pct,0.380120
2,GPPS_ICS,GP Patient Survey,2025,England,ICS,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CARE SYSTEM,gpcontactoverall,1,gpcontactoverall_1.pct,0.448645


gpps_ethnicity_distribution


,table,source,year,geography,area_geography,area_code,area_name,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,GPPS_ICS,GP Patient Survey,2025,England,ICS,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE SYSTEM,population_distribution,1,White,dv_ethnicityband_1.pct,0.845575
1,GPPS_ICS,GP Patient Survey,2025,England,ICS,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,population_distribution,1,White,dv_ethnicityband_1.pct,0.834100
2,GPPS_ICS,GP Patient Survey,2025,England,ICS,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CARE SYSTEM,population_distribution,1,White,dv_ethnicityband_1.pct,0.925890


ons_birth_characteristics_national


,table,source,year,geography,area_geography,area_code,area_name,category,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All gestational ages,total_live_births,All,All,Total live births,605060.0
1,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,"Under 22 weeks and birthweight less than 1,000g",total_live_births,All,All,Total live births,305.0
2,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,22 weeks,total_live_births,All,All,Total live births,229.0


Part 2 - Schema audit

Assess whether datasets follow a consistent analytical schema and identify structural gaps.

In [11]:
# Key columns for schema audit
KEY_COLUMNS = [
    "table",
    "source",
    "year",
    "geography",
    "area_code",
    "area_name",
    "category",
    "metric",
    "ethnicity_standard",
    "measure_raw",
    "value",
]

In [12]:
def audit_schema(dataset_name: str, dataframe: pd.DataFrame) -> pd.DataFrame:
    audit_rows = []

    for column in KEY_COLUMNS:
        column_present = column in dataframe.columns

        audit_rows.append(
            {
                "dataset_name": dataset_name,
                "column": column,
                "present": column_present,
                "dtype": str(dataframe[column].dtype) if column_present else None,
                "missing_n": int(dataframe[column].isna().sum()) if column_present else None,
                "missing_pct": float(dataframe[column].isna().mean()) if column_present else None,
                "n_unique": int(dataframe[column].nunique(dropna=True)) if column_present else None,
            }
        )

    return pd.DataFrame(audit_rows)

In [13]:
# Run schema audit across all datasets
schema_audit_frames = []

for dataset_name, dataframe in datasets.items():
    schema_audit_frames.append(audit_schema(dataset_name, dataframe))

schema_audit = pd.concat(schema_audit_frames, ignore_index=True)

print(f"Schema audit rows: {len(schema_audit):,}")
schema_audit.head(30)

Schema audit rows: 88


,dataset_name,column,present,dtype,missing_n,missing_pct,n_unique
0,fingertips_selected_indicators,table,False,None,None,None,None
1,fingertips_selected_indicators,source,False,None,None,None,None
2,fingertips_selected_indicators,year,False,None,None,None,None
3,fingertips_selected_indicators,geography,False,None,None,None,None
4,fingertips_selected_indicators,area_code,False,None,None,None,None
5,fingertips_selected_indicators,area_name,False,None,None,None,None
6,fingertips_selected_indicators,category,False,None,None,None,None
7,fingertips_selected_indicators,metric,False,None,None,None,None
8,fingertips_selected_indicators,ethnicity_standard,False,None,None,None,None
9,fingertips_selected_indicators,measure_raw,False,None,None,None,None


In [15]:
# Save schema audit
schema_audit.to_csv(PROJECT_DIR / "_notebook10a_schema_audit.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_schema_audit.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_schema_audit.csv


In [16]:
# Schema presence matrix
schema_presence = (
    schema_audit
    .pivot(index="dataset_name", columns="column", values="present")
    .fillna(False)
    .astype(bool)
)

schema_presence

column,area_code,area_name,category,ethnicity_standard,geography,measure_raw,metric,source,table,value,year
dataset_name,,,,,,,,,,,
census2021_ethnicity_nwl_analytical,False,False,False,False,False,False,False,False,False,False,False
ethnicity_facts_gp_satisfaction,False,False,False,False,False,False,False,True,False,True,False
fingertips_indicator_metadata,False,False,False,False,False,False,False,False,False,False,False
fingertips_selected_indicators,False,False,False,False,False,False,False,False,False,False,False
gla_his_borough_latest_and_trend,False,False,False,False,False,False,False,False,False,True,False
gpps_ethnicity_distribution,True,True,False,True,True,True,True,True,True,True,True
gpps_survey,True,True,True,False,True,True,True,True,True,True,True
ons_birth_characteristics_national,True,True,True,True,True,True,True,True,True,True,True


Part 3 - Coverage diagnostics & validation

Evaluate dataset completeness across time, geography, ethnicity, and metrics, and validate analytical grain.

In [17]:
# Year coverage diagnostics
def summarise_year_coverage(dataset_name: str, dataframe: pd.DataFrame) -> dict:
    summary = {
        "dataset_name": dataset_name,
        "has_year": "year" in dataframe.columns,
        "year_dtype": None,
        "year_min": None,
        "year_max": None,
        "year_n_unique": None,
        "year_values_sample": None,
    }

    if "year" not in dataframe.columns:
        return summary

    year_series = dataframe["year"].dropna()
    summary["year_dtype"] = str(dataframe["year"].dtype)
    summary["year_n_unique"] = int(year_series.nunique())

    if year_series.empty:
        return summary

    numeric_year = pd.to_numeric(year_series, errors="coerce").dropna()

    if not numeric_year.empty:
        summary["year_min"] = int(numeric_year.min())
        summary["year_max"] = int(numeric_year.max())
    else:
        unique_values = sorted(year_series.astype(str).unique().tolist())
        summary["year_values_sample"] = unique_values[:10]

    return summary

In [18]:
# Run year coverage diagnostics
year_coverage_rows = []

for dataset_name, dataframe in datasets.items():
    year_coverage_rows.append(summarise_year_coverage(dataset_name, dataframe))

year_coverage = pd.DataFrame(year_coverage_rows).sort_values("dataset_name").reset_index(drop=True)
year_coverage

,dataset_name,has_year,year_dtype,year_min,year_max,year_n_unique,year_values_sample
0,census2021_ethnicity_nwl_analytical,False,NaN,NaN,NaN,NaN,None
1,ethnicity_facts_gp_satisfaction,False,NaN,NaN,NaN,NaN,None
2,fingertips_indicator_metadata,False,NaN,NaN,NaN,NaN,None
3,fingertips_selected_indicators,False,NaN,NaN,NaN,NaN,None
4,gla_his_borough_latest_and_trend,False,NaN,NaN,NaN,NaN,None
5,gpps_ethnicity_distribution,True,int64,2025.0,2025.0,1.0,None
6,gpps_survey,True,int64,2025.0,2025.0,1.0,None
7,ons_birth_characteristics_national,True,int64,2007.0,2022.0,16.0,None


In [19]:
# Save year coverage
year_coverage.to_csv(PROJECT_DIR / "_notebook10a_year_coverage.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_year_coverage.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_year_coverage.csv


In [20]:
# Geography coverage diagnostics
def summarise_geography_coverage(dataset_name: str, dataframe: pd.DataFrame) -> dict:
    summary = {
        "dataset_name": dataset_name,
        "has_geography": "geography" in dataframe.columns,
        "has_area_code": "area_code" in dataframe.columns,
        "has_area_name": "area_name" in dataframe.columns,
        "geography_n_unique": None,
        "geography_values_sample": None,
        "area_code_n_unique": None,
        "area_name_n_unique": None,
    }

    if "geography" in dataframe.columns:
        geography_values = dataframe["geography"].dropna().astype(str).unique().tolist()
        summary["geography_n_unique"] = len(geography_values)
        summary["geography_values_sample"] = sorted(geography_values)[:10]

    if "area_code" in dataframe.columns:
        summary["area_code_n_unique"] = int(dataframe["area_code"].dropna().nunique())

    if "area_name" in dataframe.columns:
        summary["area_name_n_unique"] = int(dataframe["area_name"].dropna().nunique())

    return summary

In [21]:
# Run geography coverage diagnostics
geography_coverage_rows = []

for dataset_name, dataframe in datasets.items():
    geography_coverage_rows.append(summarise_geography_coverage(dataset_name, dataframe))

geography_coverage = (
    pd.DataFrame(geography_coverage_rows)
    .sort_values("dataset_name")
    .reset_index(drop=True)
)

geography_coverage

,dataset_name,has_geography,has_area_code,has_area_name,geography_n_unique,geography_values_sample,area_code_n_unique,area_name_n_unique
0,census2021_ethnicity_nwl_analytical,False,False,False,NaN,None,NaN,NaN
1,ethnicity_facts_gp_satisfaction,False,False,False,NaN,None,NaN,NaN
2,fingertips_indicator_metadata,False,False,False,NaN,None,NaN,NaN
3,fingertips_selected_indicators,False,False,False,NaN,None,NaN,NaN
4,gla_his_borough_latest_and_trend,False,False,False,NaN,None,NaN,NaN
5,gpps_ethnicity_distribution,True,True,True,1.0,[England],42.0,42.0
6,gpps_survey,True,True,True,1.0,[England],42.0,42.0
7,ons_birth_characteristics_national,True,True,True,1.0,[England and Wales],1.0,1.0


In [22]:
# Save geography coverage
geography_coverage.to_csv(PROJECT_DIR / "_notebook10a_geography_coverage.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_geography_coverage.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_geography_coverage.csv


In [23]:
# Ethnicity coverage diagnostics
def summarise_ethnicity_coverage(dataset_name: str, dataframe: pd.DataFrame) -> dict:
    summary = {
        "dataset_name": dataset_name,
        "has_ethnicity_standard": "ethnicity_standard" in dataframe.columns,
        "ethnicity_n_unique": None,
        "ethnicity_values_sample": None,
    }

    if "ethnicity_standard" in dataframe.columns:
        ethnicity_values = (
            dataframe["ethnicity_standard"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        summary["ethnicity_n_unique"] = len(ethnicity_values)
        summary["ethnicity_values_sample"] = sorted(ethnicity_values)[:20]

    return summary

In [24]:
# Run ethnicity coverage diagnostics
ethnicity_coverage_rows = []

for dataset_name, dataframe in datasets.items():
    ethnicity_coverage_rows.append(summarise_ethnicity_coverage(dataset_name, dataframe))

ethnicity_coverage = (
    pd.DataFrame(ethnicity_coverage_rows)
    .sort_values("dataset_name")
    .reset_index(drop=True)
)

ethnicity_coverage

,dataset_name,has_ethnicity_standard,ethnicity_n_unique,ethnicity_values_sample
0,census2021_ethnicity_nwl_analytical,False,NaN,None
1,ethnicity_facts_gp_satisfaction,False,NaN,None
2,fingertips_indicator_metadata,False,NaN,None
3,fingertips_selected_indicators,False,NaN,None
4,gla_his_borough_latest_and_trend,False,NaN,None
5,gpps_ethnicity_distribution,True,6.0,"[Asian, Black, Mixed, Not stated, Other, White]"
6,gpps_survey,False,NaN,None
7,ons_birth_characteristics_national,True,17.0,"[All, Any other Asian background, Any other Black background, Any other White background, Any other ethnic group, As..."


In [25]:
# Save ethnicity coverage
ethnicity_coverage.to_csv(PROJECT_DIR / "_notebook10a_ethnicity_coverage.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_ethnicity_coverage.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_ethnicity_coverage.csv


In [26]:
# Metric coverage diagnostics
def summarise_metric_coverage(dataset_name: str, dataframe: pd.DataFrame) -> dict:
    summary = {
        "dataset_name": dataset_name,
        "has_metric": "metric" in dataframe.columns,
        "metric_n_unique": None,
        "metric_values_sample": None,
        "has_measure_raw": "measure_raw" in dataframe.columns,
        "measure_raw_n_unique": None,
        "measure_raw_values_sample": None,
    }

    if "metric" in dataframe.columns:
        metric_values = dataframe["metric"].dropna().astype(str).unique().tolist()
        summary["metric_n_unique"] = len(metric_values)
        summary["metric_values_sample"] = sorted(metric_values)[:15]

    if "measure_raw" in dataframe.columns:
        measure_values = dataframe["measure_raw"].dropna().astype(str).unique().tolist()
        summary["measure_raw_n_unique"] = len(measure_values)
        summary["measure_raw_values_sample"] = sorted(measure_values)[:15]

    return summary

In [27]:
# Run metric coverage diagnostics
metric_coverage_rows = []

for dataset_name, dataframe in datasets.items():
    metric_coverage_rows.append(summarise_metric_coverage(dataset_name, dataframe))

metric_coverage = (
    pd.DataFrame(metric_coverage_rows)
    .sort_values("dataset_name")
    .reset_index(drop=True)
)

metric_coverage

,dataset_name,has_metric,metric_n_unique,metric_values_sample,has_measure_raw,measure_raw_n_unique,measure_raw_values_sample
0,census2021_ethnicity_nwl_analytical,False,NaN,None,False,NaN,None
1,ethnicity_facts_gp_satisfaction,False,NaN,None,False,NaN,None
2,fingertips_indicator_metadata,False,NaN,None,False,NaN,None
3,fingertips_selected_indicators,False,NaN,None,False,NaN,None
4,gla_his_borough_latest_and_trend,False,NaN,None,False,NaN,None
5,gpps_ethnicity_distribution,True,1.0,[population_distribution],True,6.0,"[dv_ethnicityband_1.pct, dv_ethnicityband_2.pct, dv_ethnicityband_3.pct, dv_ethnicityband_4.pct, dv_ethnicityband_5...."
6,gpps_survey,True,3.0,"[gpcontactoverall, healthconfidence, overallexp]",True,15.0,"[gpcontactoverall_1.pct, gpcontactoverall_2.pct, gpcontactoverall_3.pct, gpcontactoverall_4.pct, gpcontactoverall_5...."
7,ons_birth_characteristics_national,True,6.0,"[live_births, number_live_births, number_stillbirths, stillbirth_rate, total_live_births, total_stillbirths]",True,68.0,"[Live births Any other Asian background, Live births Any other Black background, Live births Any other ethnic group,..."


In [28]:
# Save metric coverage
metric_coverage.to_csv(PROJECT_DIR / "_notebook10a_metric_coverage.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_metric_coverage.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_metric_coverage.csv


In [29]:
# Consolidated coverage summary
coverage_summary = (
    dataset_registry[["dataset_name", "rows", "columns"]]
    .merge(year_coverage, on="dataset_name", how="left")
    .merge(geography_coverage, on="dataset_name", how="left")
    .merge(ethnicity_coverage, on="dataset_name", how="left")
    .merge(metric_coverage, on="dataset_name", how="left")
)

coverage_summary

,dataset_name,rows,columns,has_year,year_dtype,year_min,year_max,year_n_unique,year_values_sample,has_geography,has_area_code,has_area_name,geography_n_unique,geography_values_sample,area_code_n_unique,area_name_n_unique,has_ethnicity_standard,ethnicity_n_unique,ethnicity_values_sample,has_metric,metric_n_unique,metric_values_sample,has_measure_raw,measure_raw_n_unique,measure_raw_values_sample
0,census2021_ethnicity_nwl_analytical,64,6,False,NaN,NaN,NaN,NaN,None,False,False,False,NaN,None,NaN,NaN,False,NaN,None,False,NaN,None,False,NaN,None
1,ethnicity_facts_gp_satisfaction,21,11,False,NaN,NaN,NaN,NaN,None,False,False,False,NaN,None,NaN,NaN,False,NaN,None,False,NaN,None,False,NaN,None
2,fingertips_indicator_metadata,1008,32,False,NaN,NaN,NaN,NaN,None,False,False,False,NaN,None,NaN,NaN,False,NaN,None,False,NaN,None,False,NaN,None
3,fingertips_selected_indicators,5118,28,False,NaN,NaN,NaN,NaN,None,False,False,False,NaN,None,NaN,NaN,False,NaN,None,False,NaN,None,False,NaN,None
4,gla_his_borough_latest_and_trend,3611,14,False,NaN,NaN,NaN,NaN,None,False,False,False,NaN,None,NaN,NaN,False,NaN,None,False,NaN,None,False,NaN,None
5,gpps_ethnicity_distribution,252,12,True,int64,2025.0,2025.0,1.0,None,True,True,True,1.0,[England],42.0,42.0,True,6.0,"[Asian, Black, Mixed, Not stated, Other, White]",True,1.0,[population_distribution],True,6.0,"[dv_ethnicityband_1.pct, dv_ethnicityband_2.pct, dv_ethnicityband_3.pct, dv_ethnicityband_4.pct, dv_ethnicityband_5...."
6,gpps_survey,630,11,True,int64,2025.0,2025.0,1.0,None,True,True,True,1.0,[England],42.0,42.0,False,NaN,None,True,3.0,"[gpcontactoverall, healthconfidence, overallexp]",True,15.0,"[gpcontactoverall_1.pct, gpcontactoverall_2.pct, gpcontactoverall_3.pct, gpcontactoverall_4.pct, gpcontactoverall_5...."
7,ons_birth_characteristics_national,995,13,True,int64,2007.0,2022.0,16.0,None,True,True,True,1.0,[England and Wales],1.0,1.0,True,17.0,"[All, Any other Asian background, Any other Black background, Any other White background, Any other ethnic group, As...",True,6.0,"[live_births, number_live_births, number_stillbirths, stillbirth_rate, total_live_births, total_stillbirths]",True,68.0,"[Live births Any other Asian background, Live births Any other Black background, Live births Any other ethnic group,..."


In [30]:
# Save consolidated coverage summary
coverage_summary.to_csv(PROJECT_DIR / "_notebook10a_coverage_summary.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_coverage_summary.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_coverage_summary.csv


Part 3b - Dataset role classification

Classify datasets by analytical role to define how each contributes to the evidence base.

In [31]:
# Dataset role classification
DATASET_ROLES = {
    # Borough-level outcomes fallback
    "fingertips_selected_indicators": "outcomes_observed_fallback",

    # Metadata only
    "fingertips_indicator_metadata": "metadata_only",

    # Borough-level determinants fallback
    "gla_his_borough_latest_and_trend": "determinants_observed_fallback",

    # Borough ethnicity denominators
    "census2021_ethnicity_nwl_analytical": "population_observed",

    # National contextual evidence
    "ethnicity_facts_gp_satisfaction": "national_context",

    # ICS service experience
    "gpps_survey": "experience_observed",

    # ICS ethnicity composition
    "gpps_ethnicity_distribution": "ethnicity_context",

    # National ethnicity disparity baseline
    "ons_birth_characteristics_national": "national_disparity",
}

In [32]:
# Add dataset roles to registry
dataset_registry["dataset_role"] = dataset_registry["dataset_name"].map(DATASET_ROLES)

dataset_registry[["dataset_name", "dataset_role", "rows", "columns"]]

,dataset_name,dataset_role,rows,columns
0,census2021_ethnicity_nwl_analytical,population_observed,64,6
1,ethnicity_facts_gp_satisfaction,national_context,21,11
2,fingertips_indicator_metadata,metadata_only,1008,32
3,fingertips_selected_indicators,outcomes_observed_fallback,5118,28
4,gla_his_borough_latest_and_trend,determinants_observed_fallback,3611,14
5,gpps_ethnicity_distribution,ethnicity_context,252,12
6,gpps_survey,experience_observed,630,11
7,ons_birth_characteristics_national,national_disparity,995,13


In [33]:
# Analytical datasets (used downstream)
EXCLUDE_ROLES = {"metadata_only"}

analytical_datasets = {
    name: df
    for name, df in datasets.items()
    if DATASET_ROLES.get(name) not in EXCLUDE_ROLES
}

print("Analytical datasets used downstream:")
print(list(analytical_datasets.keys()))

Analytical datasets used downstream:
['fingertips_selected_indicators', 'gla_his_borough_latest_and_trend', 'census2021_ethnicity_nwl_analytical', 'ethnicity_facts_gp_satisfaction', 'gpps_survey', 'gpps_ethnicity_distribution', 'ons_birth_characteristics_national']


In [34]:
# Schema presence (analytical datasets only)
schema_audit_analytical = schema_audit[
    schema_audit["dataset_name"].isin(analytical_datasets.keys())
].copy()

schema_presence_analytical = (
    schema_audit_analytical
    .pivot(index="dataset_name", columns="column", values="present")
    .fillna(False)
    .astype(bool)
)

schema_presence_analytical

column,area_code,area_name,category,ethnicity_standard,geography,measure_raw,metric,source,table,value,year
dataset_name,,,,,,,,,,,
census2021_ethnicity_nwl_analytical,False,False,False,False,False,False,False,False,False,False,False
ethnicity_facts_gp_satisfaction,False,False,False,False,False,False,False,True,False,True,False
fingertips_selected_indicators,False,False,False,False,False,False,False,False,False,False,False
gla_his_borough_latest_and_trend,False,False,False,False,False,False,False,False,False,True,False
gpps_ethnicity_distribution,True,True,False,True,True,True,True,True,True,True,True
gpps_survey,True,True,True,False,True,True,True,True,True,True,True
ons_birth_characteristics_national,True,True,True,True,True,True,True,True,True,True,True


Part 3c - Structural validation & grain checks

Confirm dataset structure, inspect columns, and validate uniqueness at the intended analytical grain.

In [35]:
# Detailed structural summary for analytical datasets
def build_structural_summary(dataset_name: str, dataframe: pd.DataFrame) -> dict:
    summary = {
        "dataset_name": dataset_name,
        "rows": int(dataframe.shape[0]),
        "columns": int(dataframe.shape[1]),
        "column_names": list(dataframe.columns),
        "sample_columns": list(dataframe.columns[:10]),
    }

    return summary


structural_summary_rows = []

for dataset_name, dataframe in analytical_datasets.items():
    structural_summary_rows.append(build_structural_summary(dataset_name, dataframe))

structural_summary = (
    pd.DataFrame(structural_summary_rows)
    .sort_values("dataset_name")
    .reset_index(drop=True)
)

structural_summary

,dataset_name,rows,columns,column_names,sample_columns
0,census2021_ethnicity_nwl_analytical,64,6,"[geography_code, geography_name, analytical_ethnicity, population, borough_total, proportion]","[geography_code, geography_name, analytical_ethnicity, population, borough_total, proportion]"
1,ethnicity_facts_gp_satisfaction,21,11,"[source, dataset, geography_level, geography_name, time_period, measure_name, value, value_unit, raw_ethnicity, stan...","[source, dataset, geography_level, geography_name, time_period, measure_name, value, value_unit, raw_ethnicity, stan..."
2,fingertips_selected_indicators,5118,28,"[Indicator ID, Indicator Name, Parent Code, Parent Name, Area Code, Area Name, Area Type, Sex, Age, Category Type, C...","[Indicator ID, Indicator Name, Parent Code, Parent Name, Area Code, Area Name, Area Type, Sex, Age, Category Type]"
3,gla_his_borough_latest_and_trend,3611,14,"[time_period, borough, value, lower_ci, upper_ci, neg_error_bar, pos_error_bar, indicator_group, data_version, , Neg...","[time_period, borough, value, lower_ci, upper_ci, neg_error_bar, pos_error_bar, indicator_group, data_version, ]"
4,gpps_ethnicity_distribution,252,12,"[table, source, year, geography, area_geography, area_code, area_name, metric, ethnicity_raw, ethnicity_standard, me...","[table, source, year, geography, area_geography, area_code, area_name, metric, ethnicity_raw, ethnicity_standard]"
5,gpps_survey,630,11,"[table, source, year, geography, area_geography, area_code, area_name, metric, category, measure_raw, value]","[table, source, year, geography, area_geography, area_code, area_name, metric, category, measure_raw]"
6,ons_birth_characteristics_national,995,13,"[table, source, year, geography, area_geography, area_code, area_name, category, metric, ethnicity_raw, ethnicity_st...","[table, source, year, geography, area_geography, area_code, area_name, category, metric, ethnicity_raw]"


In [36]:
# Save structural summary
structural_summary.to_csv(PROJECT_DIR / "_notebook10a_structural_summary.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_structural_summary.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_structural_summary.csv


In [37]:
# Exact columns by analytical dataset
for dataset_name, dataframe in analytical_datasets.items():
    print("=" * 100)
    print(dataset_name)
    print("=" * 100)
    print(list(dataframe.columns))
    print()

fingertips_selected_indicators
['Indicator ID', 'Indicator Name', 'Parent Code', 'Parent Name', 'Area Code', 'Area Name', 'Area Type', 'Sex', 'Age', 'Category Type', 'Category', 'Time period', 'Value', 'Lower CI 95.0 limit', 'Upper CI 95.0 limit', 'Lower CI 99.8 limit', 'Upper CI 99.8 limit', 'Count', 'Denominator', 'Value note', 'Recent Trend', 'Compared to England value or percentiles', 'Column not used', 'Time period Sortable', 'New data', 'Compared to goal', 'Time period range', 'indicator_id']

gla_his_borough_latest_and_trend
['time_period', 'borough', 'value', 'lower_ci', 'upper_ci', 'neg_error_bar', 'pos_error_bar', 'indicator_group', 'data_version', '', 'Negative error bar', 'Positive error bar', 'unnamed', 'unnamed__2']

census2021_ethnicity_nwl_analytical
['geography_code', 'geography_name', 'analytical_ethnicity', 'population', 'borough_total', 'proportion']

ethnicity_facts_gp_satisfaction
['source', 'dataset', 'geography_level', 'geography_name', 'time_period', 'measure_n

In [38]:
# Candidate grain keys by dataset
CANDIDATE_GRAIN_KEYS = {
    "fingertips_selected_indicators": [
        "Indicator ID",
        "Area Code",
        "Area Name",
        "Time period",
        "Sex",
        "Age",
        "Category Type",
        "Category",
    ],
    "gla_his_borough_latest_and_trend": [
        "borough",
        "time_period",
        "indicator_group",
        "data_version",
    ],
    "census2021_ethnicity_nwl_analytical": [
        "geography_code",
        "geography_name",
        "analytical_ethnicity",
    ],
    "ethnicity_facts_gp_satisfaction": [
        "dataset",
        "geography_level",
        "geography_name",
        "time_period",
        "measure_name",
        "standard_ethnicity",
    ],
    "gpps_survey": [
        "table",
        "year",
        "area_geography",
        "area_code",
        "area_name",
        "metric",
        "category",
        "measure_raw",
    ],
    "gpps_ethnicity_distribution": [
        "table",
        "year",
        "area_geography",
        "area_code",
        "area_name",
        "metric",
        "ethnicity_standard",
        "measure_raw",
    ],
    "ons_birth_characteristics_national": [
        "table",
        "year",
        "area_geography",
        "area_code",
        "area_name",
        "category",
        "metric",
        "ethnicity_standard",
        "measure_raw",
    ],
}

In [39]:
# Duplicate check function
def check_duplicate_grain(dataset_name: str, dataframe: pd.DataFrame, key_columns: list[str]) -> dict:
    available_keys = [column for column in key_columns if column in dataframe.columns]

    result = {
        "dataset_name": dataset_name,
        "candidate_keys": key_columns,
        "available_keys": available_keys,
        "all_keys_available": len(available_keys) == len(key_columns),
        "rows_total": int(len(dataframe)),
        "rows_distinct_keys": None,
        "duplicate_rows": None,
        "duplicate_pct": None,
    }

    if not available_keys:
        return result

    distinct_key_rows = dataframe[available_keys].drop_duplicates().shape[0]
    duplicate_rows = len(dataframe) - distinct_key_rows

    result["rows_distinct_keys"] = int(distinct_key_rows)
    result["duplicate_rows"] = int(duplicate_rows)
    result["duplicate_pct"] = float(duplicate_rows / len(dataframe)) if len(dataframe) > 0 else 0.0

    return result

In [40]:
# Run duplicate checks
duplicate_check_rows = []

for dataset_name, dataframe in analytical_datasets.items():
    candidate_keys = CANDIDATE_GRAIN_KEYS.get(dataset_name, [])
    duplicate_check_rows.append(
        check_duplicate_grain(dataset_name, dataframe, candidate_keys)
    )

duplicate_checks = (
    pd.DataFrame(duplicate_check_rows)
    .sort_values("dataset_name")
    .reset_index(drop=True)
)

duplicate_checks

,dataset_name,candidate_keys,available_keys,all_keys_available,rows_total,rows_distinct_keys,duplicate_rows,duplicate_pct
0,census2021_ethnicity_nwl_analytical,"[geography_code, geography_name, analytical_ethnicity]","[geography_code, geography_name, analytical_ethnicity]",True,64,64,0,0.000000
1,ethnicity_facts_gp_satisfaction,"[dataset, geography_level, geography_name, time_period, measure_name, standard_ethnicity]","[dataset, geography_level, geography_name, time_period, measure_name, standard_ethnicity]",True,21,21,0,0.000000
2,fingertips_selected_indicators,"[Indicator ID, Area Code, Area Name, Time period, Sex, Age, Category Type, Category]","[Indicator ID, Area Code, Area Name, Time period, Sex, Age, Category Type, Category]",True,5118,5118,0,0.000000
3,gla_his_borough_latest_and_trend,"[borough, time_period, indicator_group, data_version]","[borough, time_period, indicator_group, data_version]",True,3611,3545,66,0.018277
4,gpps_ethnicity_distribution,"[table, year, area_geography, area_code, area_name, metric, ethnicity_standard, measure_raw]","[table, year, area_geography, area_code, area_name, metric, ethnicity_standard, measure_raw]",True,252,252,0,0.000000
5,gpps_survey,"[table, year, area_geography, area_code, area_name, metric, category, measure_raw]","[table, year, area_geography, area_code, area_name, metric, category, measure_raw]",True,630,630,0,0.000000
6,ons_birth_characteristics_national,"[table, year, area_geography, area_code, area_name, category, metric, ethnicity_standard, measure_raw]","[table, year, area_geography, area_code, area_name, category, metric, ethnicity_standard, measure_raw]",True,995,995,0,0.000000


In [41]:
# Save duplicate checks
duplicate_checks.to_csv(PROJECT_DIR / "_notebook10a_duplicate_checks.csv", index=False)
print("Saved:", PROJECT_DIR / "_notebook10a_duplicate_checks.csv")

Saved: c:\Users\OlenaManziuk\OneDrive - Third Sector Together North West London\Documents\nwl_health_inequalities\_notebook10a_duplicate_checks.csv


In [42]:
# Borough coverage quick inspection
for dataset_name in [
    "fingertips_selected_indicators",
    "gla_his_borough_latest_and_trend",
    "census2021_ethnicity_nwl_analytical",
]:
    if dataset_name not in analytical_datasets:
        continue

    dataframe = analytical_datasets[dataset_name]

    print("=" * 100)
    print(dataset_name)
    print("=" * 100)

    if "Area Name" in dataframe.columns:
        print("Area Name sample:")
        print(sorted(dataframe["Area Name"].dropna().astype(str).unique().tolist())[:20])

    if "borough" in dataframe.columns:
        print("borough sample:")
        print(sorted(dataframe["borough"].dropna().astype(str).unique().tolist())[:20])

    if "geography_name" in dataframe.columns:
        print("geography_name sample:")
        print(sorted(dataframe["geography_name"].dropna().astype(str).unique().tolist())[:20])

    print()

fingertips_selected_indicators
Area Name sample:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']

gla_his_borough_latest_and_trend
borough sample:
['Area Name', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington']

census2021_ethnicity_nwl_analytical
geography_name sample:
['Brent', 'Ealing', 'Hammersmith and Fulham', 'Harrow', 'Hillingdon', 'Hounslow', 'Kensington and Chelsea', 'Westminster']



Part 3d - Data quality fixes (GLA)

Resolve identified data quality issues to ensure datasets are analysis-ready.

In [43]:
# Clean GLA borough dataset (remove bad rows and columns)
gla_df = analytical_datasets["gla_his_borough_latest_and_trend"].copy()

# Remove invalid borough rows
gla_df = gla_df[
    gla_df["borough"].notna()
    & (gla_df["borough"].astype(str).str.strip() != "")
    & (gla_df["borough"] != "Area Name")
]

# Drop junk columns
cols_to_drop = ["", "unnamed", "unnamed__2", "Negative error bar", "Positive error bar"]
cols_to_drop = [col for col in cols_to_drop if col in gla_df.columns]

gla_df = gla_df.drop(columns=cols_to_drop)

# Update back into analytical datasets
analytical_datasets["gla_his_borough_latest_and_trend"] = gla_df

print("Cleaned GLA dataset shape:", gla_df.shape)

Cleaned GLA dataset shape: (3609, 9)


In [44]:
# Re-check duplicates for cleaned GLA dataset
check_duplicate_grain(
    "gla_his_borough_latest_and_trend",
    analytical_datasets["gla_his_borough_latest_and_trend"],
    CANDIDATE_GRAIN_KEYS["gla_his_borough_latest_and_trend"]
)

{'dataset_name': 'gla_his_borough_latest_and_trend',
 'candidate_keys': ['borough',
  'time_period',
  'indicator_group',
  'data_version'],
 'available_keys': ['borough',
  'time_period',
  'indicator_group',
  'data_version'],
 'all_keys_available': True,
 'rows_total': 3609,
 'rows_distinct_keys': 3543,
 'duplicate_rows': 66,
 'duplicate_pct': 0.01828761429758936}

In [45]:
# Inspect duplicate rows in GLA dataset
gla_df = analytical_datasets["gla_his_borough_latest_and_trend"]

key_cols = CANDIDATE_GRAIN_KEYS["gla_his_borough_latest_and_trend"]

duplicate_mask = gla_df.duplicated(subset=key_cols, keep=False)

gla_duplicates = gla_df.loc[duplicate_mask].copy()

print("Number of duplicate rows:", len(gla_duplicates))

gla_duplicates.sort_values(key_cols).head(20)

Number of duplicate rows: 132


,time_period,borough,value,lower_ci,upper_ci,neg_error_bar,pos_error_bar,indicator_group,data_version
17,2021 - 23,Barking and Dagenham,63.45,59.19,67.72,4.26,4.27,1. HLE male,latest
44,2021 - 23,Barking and Dagenham,63.45,59.19,67.72,4.26,4.27,1. HLE male,latest
778,2021 - 23,Barking and Dagenham,63.52,59.22,67.82,4.30,4.30,2. HLE female,latest
804,2021 - 23,Barking and Dagenham,63.52,59.22,67.82,4.30,4.30,2. HLE female,latest
11,2021 - 23,Barnet,65.02,61.41,68.63,3.61,3.61,1. HLE male,latest
55,2021 - 23,Barnet,65.02,61.41,68.63,3.61,3.61,1. HLE male,latest
772,2021 - 23,Barnet,64.62,60.22,69.03,4.40,4.41,2. HLE female,latest
815,2021 - 23,Barnet,64.62,60.22,69.03,4.40,4.41,2. HLE female,latest
13,2021 - 23,Bexley,64.20,60.28,68.12,3.92,3.92,1. HLE male,latest
66,2021 - 23,Bexley,64.20,60.28,68.12,3.92,3.92,1. HLE male,latest


In [46]:
# Duplicate breakdown
gla_duplicates.groupby(key_cols).size().reset_index(name="count").sort_values("count", ascending=False).head(10)

,borough,time_period,indicator_group,data_version,count
0,Barking and Dagenham,2021 - 23,1. HLE male,latest,2
1,Barking and Dagenham,2021 - 23,2. HLE female,latest,2
2,Barnet,2021 - 23,1. HLE male,latest,2
3,Barnet,2021 - 23,2. HLE female,latest,2
4,Bexley,2021 - 23,1. HLE male,latest,2
5,Bexley,2021 - 23,2. HLE female,latest,2
6,Brent,2021 - 23,1. HLE male,latest,2
7,Brent,2021 - 23,2. HLE female,latest,2
8,Bromley,2021 - 23,1. HLE male,latest,2
9,Bromley,2021 - 23,2. HLE female,latest,2


In [47]:
# Remove exact duplicate rows from cleaned GLA dataset
gla_df = analytical_datasets["gla_his_borough_latest_and_trend"].copy()

rows_before = len(gla_df)

gla_df = gla_df.drop_duplicates()

rows_after = len(gla_df)

analytical_datasets["gla_his_borough_latest_and_trend"] = gla_df

print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Duplicates removed: {rows_before - rows_after}")

Rows before: 3609
Rows after: 3543
Duplicates removed: 66


In [48]:
# Re-check duplicate grain after exact deduplication
check_duplicate_grain(
    "gla_his_borough_latest_and_trend",
    analytical_datasets["gla_his_borough_latest_and_trend"],
    CANDIDATE_GRAIN_KEYS["gla_his_borough_latest_and_trend"]
)

{'dataset_name': 'gla_his_borough_latest_and_trend',
 'candidate_keys': ['borough',
  'time_period',
  'indicator_group',
  'data_version'],
 'available_keys': ['borough',
  'time_period',
  'indicator_group',
  'data_version'],
 'all_keys_available': True,
 'rows_total': 3543,
 'rows_distinct_keys': 3543,
 'duplicate_rows': 0,
 'duplicate_pct': 0.0}

Final comments:

Data has been audited, validated, and cleaned. Datasets are now ready for harmonisation and integration in Notebook 10B.

Analytical position after dataset engineering work:

- borough names checked
- bad header row removed
- junk columns removed
- exact duplicates removed
- expected grain validated

The next part will focus on:

- harmonisation scaffolding
- standardised borough fields
- standardised ethnicity fields
- standardised output tables for downstream modelling